
# Decision-certified BO with a genuinely nonlinear PDE-conditioned GP

This notebook stress-tests Decision-Equivalent / Decision-Certified Conditioning on a **nonlinear semilinear reaction-diffusion PDE**

\[
f-\kappa\Delta f+\lambda\sin f=s.
\]

The PDE residual itself is nonlinear in the latent field \(f\). We then use a robust non-Gaussian residual likelihood,

\[
e_j(f)=\gamma\log\cosh(r_j(f)/\tau),
\]

so the conditioned GP posterior is genuinely non-Gaussian for two independent reasons:

1. \(r_j(f)\) contains \(\sin f\);
2. the residual likelihood is nonquadratic.

The experiment asks a decision-side question rather than a PDE-solving question:

> How many nonlinear PDE factors must we actually evaluate, and how accurately must we sample the resulting non-Gaussian posterior, before the next expected-improvement action is certified?

The default run should reproduce approximately:

- 576 total PDE factors;
- 40 active factors at certification;
- \(\epsilon \approx 0.048\) EI units;
- ~84% GP-reference importance-sampling ESS for the sparse target;
- ~6% GP-reference ESS for the full target;
- ~59% ESS using a Laplace-preconditioned full-target proposal;
- the same BO action under DEC and held-out full-posterior validation.



## 1. Discrete nonlinear PDE

Using unit grid spacing,

\[
(1+4\kappa) f_{ij}
-\kappa \sum_{(k,\ell)\in\mathcal N(i,j)}f_{k\ell}
+\lambda\sin f_{ij}
-s_{ij}=0.
\]

After dividing by \(1+4\kappa\), write

\[
r_{ij}(f)
=
f_{ij}
-c\sum_{(k,\ell)\in\mathcal N(i,j)}f_{k\ell}
+\eta\sin f_{ij}
-b_{ij}.
\]

We use \(c=0.12,\eta=0.25\), corresponding approximately to

\[
\kappa=0.231,\qquad \lambda=0.481.
\]

A manufactured field supplies \(b_{ij}\), so it satisfies the discrete nonlinear PDE exactly.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.linalg as la
import scipy.stats as st
from pathlib import Path
import gc

OUT = Path.cwd() / "dec_nonlinear_pde_outputs"
OUT.mkdir(exist_ok=True)

N = 24
GAMMA = 0.08
TAU = 0.30
C = 0.12
ETA = 0.25

Q0 = 3.5
QL = 0.6

Y_BEST = 0.55
N_PARTICLES = 6000
FULL_SAMPLES = 6000

BATCH_ADD = 10
EPS_TARGET = 0.060
DELTA_MC = 0.05
SEED = 911

PEAK_SEP = 5.0
PEAK_SIGMA = 1.3
LEFT_AMP = 1.0
RIGHT_AMP = 0.96
RIGHT_BIAS = 0.05

KAPPA = C/(1-4*C)
LAMBDA = ETA/(1-4*C)

print(f"kappa ≈ {KAPPA:.3f}")
print(f"lambda ≈ {LAMBDA:.3f}")



## 2. Gaussian reference and manufactured nonlinear physics

The Gaussian reference

\[
P_0(df)=\mathcal N(m,Q^{-1})
\]

should be interpreted as the ordinary data-conditioned GP at the current BO iteration.

The PDE factors are *additional* scientific information. The BO search has already localized to a central trust region, while PDE residual conditions exist over the full field.


In [ ]:

def build_problem(n, right_bias=RIGHT_BIAS):
    coords = np.arange(n) - (n-1)/2
    X,Y = np.meshgrid(coords,coords,indexing="ij")

    g_left = np.exp(
        -((X+PEAK_SEP/2)**2 + Y**2)/(2*PEAK_SIGMA**2)
    )
    g_right = np.exp(
        -((X-PEAK_SEP/2)**2 + Y**2)/(2*PEAK_SIGMA**2)
    )

    truth = LEFT_AMP*g_left + RIGHT_AMP*g_right
    mean = LEFT_AMP*g_left + (RIGHT_AMP+right_bias)*g_right

    d = n*n
    idx = lambda i,j: i*n+j

    Q = np.zeros((d,d))
    degree = np.zeros(d)

    for i in range(n):
        for j in range(n):
            u = idx(i,j)
            for di,dj in [(1,0),(0,1)]:
                ii,jj = i+di,j+dj
                if ii<n and jj<n:
                    v = idx(ii,jj)
                    degree[u] += 1
                    degree[v] += 1

    np.fill_diagonal(Q,Q0+QL*degree)

    for i in range(n):
        for j in range(n):
            u = idx(i,j)
            for di,dj in [(1,0),(0,1)]:
                ii,jj = i+di,j+dj
                if ii<n and jj<n:
                    v = idx(ii,jj)
                    Q[u,v] -= QL
                    Q[v,u] -= QL

    factors = []

    for i in range(n):
        for j in range(n):
            center = idx(i,j)
            nbrs = []

            for ii,jj in [(i-1,j),(i+1,j),(i,j-1),(i,j+1)]:
                if 0 <= ii < n and 0 <= jj < n:
                    nbrs.append(idx(ii,jj))

            nbr_truth = sum(
                truth[np.unravel_index(k,(n,n))]
                for k in nbrs
            )

            b = (
                truth[i,j]
                - C*nbr_truth
                + ETA*np.sin(truth[i,j])
            )

            factors.append({
                "center": center,
                "nbrs": np.asarray(nbrs,dtype=int),
                "inds": np.asarray([center]+nbrs,dtype=int),
                "b": float(b),
                "site": (i,j),
            })

    return coords,truth,mean,Q,factors

coords,truth,mean_field,Q,factors = build_problem(N)

center = (N-1)/2
ACTION_I = range(int(center)-6,int(center)+7)
ACTION_J = range(int(center)-4,int(center)+5)
ACTION_IDS = np.array(
    [i*N+j for i in ACTION_I for j in ACTION_J],
    dtype=int,
)

plt.figure(figsize=(6.5,5.0))
plt.imshow(truth,origin="lower")
plt.scatter(
    [j for i in ACTION_I for j in ACTION_J],
    [i for i in ACTION_I for j in ACTION_J],
    s=6,alpha=0.35,label="BO trust region",
)
plt.xlabel("grid j")
plt.ylabel("grid i")
plt.title("Manufactured nonlinear reaction-diffusion field")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Field dimension: {N*N}")
print(f"PDE factors: {len(factors)}")
print(f"BO actions: {len(ACTION_IDS)}")



## 3. Why a rigorous influence bound still exists

For one nonlinear residual,

\[
r_j(f)
=
f_0-c\sum_{k\in\mathcal N(j)}f_k+\eta\sin f_0-b_j,
\]

\[
\left|\frac{\partial r_j}{\partial f_0}\right|
=
|1+\eta\cos f_0|
\le 1+\eta,
\]

\[
\left|\frac{\partial r_j}{\partial f_k}\right|=c,
\]

and

\[
\left|\frac{\partial^2r_j}{\partial f_0^2}\right|
=
|\eta\sin f_0|
\le\eta.
\]

For

\[
e_j=\gamma\log\cosh(r_j/\tau),
\]

\[
\nabla^2e_j
=
\frac{\gamma}{\tau^2}\operatorname{sech}^2(r_j/\tau)
\nabla r_j\nabla r_j^\top
+
\frac{\gamma}{\tau}\tanh(r_j/\tau)\nabla^2r_j.
\]

The first term is positive semidefinite. The only possibly negative curvature is at the residual center and is bounded below by

\[
-\frac{\gamma\eta}{\tau}.
\]

This lets us construct a conservative influence matrix \(A\) valid for every intermediate sparse/full conditioned posterior.


In [ ]:

def build_influence(Q,factors):
    d = Q.shape[0]

    # Worst possible negative center curvature from nonlinear residual.
    rho = np.diag(Q).copy() - GAMMA*ETA/TAU

    kappa = np.abs(Q.copy())
    np.fill_diagonal(kappa,0.0)

    L_factor = np.zeros((len(factors),d))

    for j,F in enumerate(factors):
        inds = F["inds"]

        dr = np.full(len(inds),C)
        dr[0] = 1+ETA

        L_factor[j,inds] = (GAMMA/TAU)*dr

        for p,u in enumerate(inds):
            for q,v in enumerate(inds):
                if u != v:
                    kappa[u,v] += (
                        GAMMA/TAU**2
                        * dr[p]*dr[q]
                    )

    A = np.diag(rho)-kappa
    return A,L_factor

A,L_FACTOR = build_influence(Q,factors)
lambda_min_A = np.linalg.eigvalsh(A)[0]
assert lambda_min_A > 0

A_FACTOR = la.cho_factor(
    A,lower=True,check_finite=False
)

print(f"lambda_min(A) = {lambda_min_A:.4f} > 0")



## 4. Proper non-Gaussian inference on the active target

For active factor set \(S\),

\[
\pi_S(df)
\propto
P_0(df)
\exp[-E_S(f)].
\]

We begin with the cheapest sampler: iid draws from \(P_0\), reweighted by

\[
w_m=\exp[-E_S(f^{(m)})].
\]

This is self-normalized importance sampling from the actual non-Gaussian posterior; no Gaussian posterior approximation is made.

Only activated factor energies are evaluated.

The inference-error module estimates the EI-gap Monte Carlo error directly with common particles.


In [ ]:

rng = np.random.default_rng(SEED)
L = np.linalg.cholesky(Q)
z = rng.standard_normal((N_PARTICLES,N*N))

particles = (
    mean_field.ravel()
    + la.solve_triangular(L.T,z.T,lower=False).T
).astype(np.float32)

EI = np.maximum(
    particles[:,ACTION_IDS]-Y_BEST,
    0,
).astype(np.float32)

factor_cache = {}

def factor_energy(j):
    if j not in factor_cache:
        F = factors[j]

        y0 = particles[:,F["center"]]
        nbrsum = particles[:,F["nbrs"]].sum(axis=1)

        residual = (
            y0
            - C*nbrsum
            + ETA*np.sin(y0)
            - F["b"]
        )

        factor_cache[j] = (
            GAMMA*np.log(np.cosh(residual/TAU))
        ).astype(np.float32)

    return factor_cache[j]

zcrit = st.norm.ppf(
    1-DELTA_MC/(2*len(ACTION_IDS))
)

def sparse_inference(active):
    if active:
        E = np.zeros(N_PARTICLES,dtype=np.float32)
        for j in active:
            E += factor_energy(j)
        logw = -E.astype(float)
    else:
        logw = np.zeros(N_PARTICLES)

    logw -= logw.max()
    w = np.exp(logw)
    wn = w/w.sum()

    ess = (
        w.sum()**2/(w@w)
    )/N_PARTICLES

    acq = wn@EI
    leader_local = int(np.argmax(acq))
    leader_id = int(ACTION_IDS[leader_local])

    Fgap = EI-EI[:,leader_local,None]
    gap = wn@Fgap

    phi = (
        w[:,None]*(Fgap-gap[None,:])
        / w.mean()
    )

    B_mc = (
        zcrit*phi.std(axis=0,ddof=1)
        / np.sqrt(N_PARTICLES)
    )

    return acq,gap,B_mc,float(ess),leader_id,leader_local



## 5. Adaptive DEC loop

The optimistic challenger score is

\[
C_S(x)
=
\widehat\alpha_S(x)
-\widehat\alpha_S(\widehat x)
+B_{\rm struct}(x,\widehat x)
+B_{\rm MC}(x,\widehat x).
\]

If its maximum exceeds the target tolerance, rank omitted PDE factors by their certified contribution to the current leader–challenger comparison and activate the top batch.


In [ ]:

active = set()
history = []
activation_batches = []

for iteration in range(50):
    acq,gap,B_mc,ess,leader_id,leader_local = sparse_inference(active)

    omitted = [
        j for j in range(len(factors))
        if j not in active
    ]

    h_omitted = (
        L_FACTOR[omitted].sum(axis=0)
        if omitted else np.zeros(N*N)
    )

    w_struct = la.cho_solve(
        A_FACTOR,h_omitted,check_finite=False
    )

    B_struct = (
        w_struct[ACTION_IDS]
        + w_struct[leader_id]
    )
    B_struct[leader_local] = 0

    envelope = gap+B_struct+B_mc
    envelope[leader_local] = 0

    challenger_local = int(np.argmax(envelope))
    challenger_id = int(ACTION_IDS[challenger_local])
    epsilon = float(envelope[challenger_local])

    li,lj = np.unravel_index(leader_id,(N,N))
    ci,cj = np.unravel_index(challenger_id,(N,N))

    history.append({
        "iteration": iteration,
        "active_M": len(active),
        "leader_i": li,
        "leader_j": lj,
        "challenger_i": ci,
        "challenger_j": cj,
        "epsilon_total": epsilon,
        "challenger_gap": float(gap[challenger_local]),
        "B_struct": float(B_struct[challenger_local]),
        "B_MC": float(B_mc[challenger_local]),
        "sparse_IS_ESS_fraction": ess,
    })

    if epsilon <= EPS_TARGET:
        break

    d = np.zeros(N*N)
    d[leader_id] = 1
    d[challenger_id] += 1

    v = la.cho_solve(
        A_FACTOR,d,check_finite=False
    )

    contribution = L_FACTOR@v

    ranked = sorted(
        omitted,
        key=lambda j: contribution[j],
        reverse=True,
    )

    to_add = ranked[:BATCH_ADD]
    activation_batches.append(to_add)
    active.update(to_add)

history_df = pd.DataFrame(history)
history_df


In [ ]:

final = history_df.iloc[-1]

print(f"Active factors: {len(active)} / {len(factors)}")
print(f"epsilon = {final.epsilon_total:.4f}")
print(f"  structural = {final.B_struct:.4f}")
print(f"  inference  = {final.B_MC:.4f}")
print(f"  sparse gap = {final.challenger_gap:.4f}")
print(f"sparse GP-IS ESS = {final.sparse_IS_ESS_fraction:.1%}")

plt.figure(figsize=(7.2,4.7))
plt.plot(
    history_df["active_M"],
    history_df["epsilon_total"],
    marker="o",label="total certificate",
)
plt.plot(
    history_df["active_M"],
    history_df["B_struct"],
    marker="s",label="structural",
)
plt.plot(
    history_df["active_M"],
    history_df["B_MC"],
    marker="^",label="inference",
)
plt.axhline(
    EPS_TARGET,linestyle="--",
    label=f"target={EPS_TARGET}",
)
plt.xlabel("Active nonlinear-PDE factors M")
plt.ylabel("EI units")
plt.title("Decision certificate")
plt.legend()
plt.tight_layout()
plt.show()



## 6. Held-out full-posterior validation

We now evaluate **all** nonlinear PDE factors only as a post-hoc check.

First try the same GP-reference importance proposal. Then build a standard Laplace-preconditioned Gaussian proposal around the full target MAP and correct it using exact importance weights.

This is deliberately modular sampling machinery, not a claimed new sampler.


In [ ]:

def all_factor_energy(samples):
    E = np.zeros(samples.shape[0])

    for F in factors:
        y0 = samples[:,F["center"]]
        nbrsum = samples[:,F["nbrs"]].sum(axis=1)

        residual = (
            y0-C*nbrsum
            + ETA*np.sin(y0)
            - F["b"]
        )

        E += GAMMA*np.log(np.cosh(residual/TAU))

    return E

E_full_ref = all_factor_energy(particles)

logw = -E_full_ref
logw -= logw.max()
w_ref = np.exp(logw)

full_GP_ESS = (
    w_ref.sum()**2/(w_ref@w_ref)
)/N_PARTICLES

print(f"Full-target GP-reference IS ESS = {full_GP_ESS:.1%}")


In [ ]:

def target_value_grad_hess(y,need_hess=True):
    dy = y-mean_field.ravel()

    value = 0.5*dy@(Q@dy)
    grad = Q@dy
    H = Q.copy() if need_hess else None

    for F in factors:
        inds = F["inds"]
        center_id = F["center"]
        nbrs = F["nbrs"]

        y0 = y[center_id]

        residual = (
            y0
            - C*y[nbrs].sum()
            + ETA*np.sin(y0)
            - F["b"]
        )

        t = np.tanh(residual/TAU)

        dr = np.full(len(inds),-C)
        dr[0] = 1+ETA*np.cos(y0)

        value += GAMMA*np.log(np.cosh(residual/TAU))
        grad[inds] += (GAMMA/TAU)*t*dr

        if need_hess:
            H[np.ix_(inds,inds)] += (
                GAMMA/TAU**2
                * (1-t*t)
                * np.outer(dr,dr)
            )

            H[center_id,center_id] += (
                GAMMA/TAU
                * t
                * (-ETA*np.sin(y0))
            )

    return value,grad,H

y_map = mean_field.ravel().copy()

for _ in range(12):
    value,grad,H = target_value_grad_hess(y_map,True)

    if np.linalg.norm(grad,np.inf) < 1e-8:
        break

    step = np.linalg.solve(H,grad)
    directional = grad@step
    step_size = 1.0

    for _ in range(15):
        candidate = y_map-step_size*step
        candidate_value,_,_ = target_value_grad_hess(candidate,False)

        if candidate_value <= value-1e-4*step_size*directional:
            break

        step_size *= 0.5

    y_map = candidate

_,_,H = target_value_grad_hess(y_map,True)

INFLATION = 1.10
rng = np.random.default_rng(1801)

Lh = np.linalg.cholesky(H/INFLATION)
z = rng.standard_normal((FULL_SAMPLES,N*N))

full_samples = (
    y_map
    + la.solve_triangular(Lh.T,z.T,lower=False).T
).astype(np.float32)

del z
gc.collect()

logw = np.empty(FULL_SAMPLES)

for start in range(0,FULL_SAMPLES,400):
    stop = min(FULL_SAMPLES,start+400)
    sb = full_samples[start:stop].astype(float)

    d_map = sb-y_map
    logq = -0.5*np.einsum(
        "bi,ij,bj->b",
        d_map,H/INFLATION,d_map,
        optimize=True,
    )

    d_prior = sb-mean_field.ravel()
    logtarget = -0.5*np.einsum(
        "bi,ij,bj->b",
        d_prior,Q,d_prior,
        optimize=True,
    )

    logtarget -= all_factor_energy(sb)
    logw[start:stop] = logtarget-logq

logw -= logw.max()
w_full = np.exp(logw)

full_Laplace_ESS = (
    w_full.sum()**2/(w_full@w_full)
)/FULL_SAMPLES

wn_full = w_full/w_full.sum()

full_EI = np.maximum(
    full_samples[:,ACTION_IDS]-Y_BEST,
    0,
)

full_acq = wn_full@full_EI
sparse_acq,*_ = sparse_inference(active)

full_local = int(np.argmax(full_acq))
dec_local = int(np.argmax(sparse_acq))

full_id = int(ACTION_IDS[full_local])
dec_id = int(ACTION_IDS[dec_local])

regret = float(
    full_acq[full_local]
    - full_acq[dec_local]
)

print(f"Full-target Laplace IS ESS = {full_Laplace_ESS:.1%}")
print("DEC action:",np.unravel_index(dec_id,(N,N)))
print("Full action:",np.unravel_index(full_id,(N,N)))
print(f"Observed EI regret = {regret:.6f}")
print(f"Certificate = {float(final.epsilon_total):.6f}")



## 7. Expanding-domain test

The decisive test is whether the amount of physics needed for the **decision** remains local as the amount of global physics grows.

The helper below repeats the same central BO problem on larger lattices. The default study uses fewer particles for speed, because the important quantity is the scaling trend.


In [ ]:

def run_scaling_case(n,n_particles,seed,eps_target=0.075):
    coords_s,truth_s,mean_s,Q_s,factors_s = build_problem(
        n,right_bias=0.05
    )

    A_s,L_s = build_influence(Q_s,factors_s)
    A_s_factor = la.cho_factor(
        A_s,lower=True,check_finite=False
    )

    rng = np.random.default_rng(seed)
    L = np.linalg.cholesky(Q_s)
    z = rng.standard_normal((n_particles,n*n))

    samples = (
        mean_s.ravel()
        + la.solve_triangular(L.T,z.T,lower=False).T
    ).astype(np.float32)

    center_s = (n-1)/2
    ii_s = range(max(0,int(center_s)-6),min(n,int(center_s)+7))
    jj_s = range(max(0,int(center_s)-4),min(n,int(center_s)+5))

    action_ids = np.array(
        [i*n+j for i in ii_s for j in jj_s],
        dtype=int,
    )

    utility = np.maximum(
        samples[:,action_ids]-Y_BEST,
        0,
    ).astype(np.float32)

    zc = st.norm.ppf(
        1-DELTA_MC/(2*len(action_ids))
    )

    cache = {}

    def energy_j(j):
        if j not in cache:
            F = factors_s[j]
            y0 = samples[:,F["center"]]
            nbrsum = samples[:,F["nbrs"]].sum(axis=1)

            residual = (
                y0-C*nbrsum
                + ETA*np.sin(y0)
                - F["b"]
            )

            cache[j] = (
                GAMMA*np.log(np.cosh(residual/TAU))
            ).astype(np.float32)

        return cache[j]

    active_s = set()

    for _ in range(50):
        if active_s:
            E = np.zeros(n_particles,dtype=np.float32)
            for j in active_s:
                E += energy_j(j)
            lw = -E.astype(float)
        else:
            lw = np.zeros(n_particles)

        lw -= lw.max()
        w = np.exp(lw)
        wn = w/w.sum()

        sparse_ess = (
            w.sum()**2/(w@w)
        )/n_particles

        acq = wn@utility
        leader_local = int(np.argmax(acq))
        leader_id = int(action_ids[leader_local])

        Fgap = utility-utility[:,leader_local,None]
        gap = wn@Fgap

        phi = (
            w[:,None]*(Fgap-gap[None,:])
            / w.mean()
        )

        B_mc = (
            zc*phi.std(axis=0,ddof=1)
            / np.sqrt(n_particles)
        )

        omitted = [
            j for j in range(len(factors_s))
            if j not in active_s
        ]

        h = (
            L_s[omitted].sum(axis=0)
            if omitted else np.zeros(n*n)
        )

        w_struct = la.cho_solve(
            A_s_factor,h,check_finite=False
        )

        B_struct = (
            w_struct[action_ids]
            + w_struct[leader_id]
        )
        B_struct[leader_local] = 0

        envelope = gap+B_struct+B_mc
        envelope[leader_local] = 0

        challenger_local = int(np.argmax(envelope))
        challenger_id = int(action_ids[challenger_local])
        epsilon = float(envelope[challenger_local])

        if epsilon <= eps_target:
            break

        d = np.zeros(n*n)
        d[leader_id] = 1
        d[challenger_id] += 1

        v = la.cho_solve(
            A_s_factor,d,check_finite=False
        )

        contribution = L_s@v

        ranked = sorted(
            omitted,
            key=lambda j: contribution[j],
            reverse=True,
        )

        active_s.update(ranked[:BATCH_ADD])

    # Post-hoc full-target GP-reference ESS.
    E_full = np.zeros(n_particles)

    for F in factors_s:
        y0 = samples[:,F["center"]]
        nbrsum = samples[:,F["nbrs"]].sum(axis=1)

        residual = (
            y0-C*nbrsum
            + ETA*np.sin(y0)
            - F["b"]
        )

        E_full += GAMMA*np.log(np.cosh(residual/TAU))

    lw_full = -E_full
    lw_full -= lw_full.max()
    wf = np.exp(lw_full)

    full_ess = (
        wf.sum()**2/(wf@wf)
    )/n_particles

    return {
        "grid_n": n,
        "N": len(factors_s),
        "M": len(active_s),
        "epsilon": epsilon,
        "sparse_ESS": sparse_ess,
        "full_ESS": full_ess,
    }

rows = []

for n_case,p_case in [
    (18,3000),
    (24,3000),
    (30,2500),
    (36,2200),
    (40,2000),
]:
    rows.append(
        run_scaling_case(
            n_case,p_case,2100+n_case
        )
    )

scaling_df = pd.DataFrame(rows)
scaling_df


In [ ]:

plt.figure(figsize=(7.2,4.7))
plt.plot(
    scaling_df["N"],
    scaling_df["M"],
    marker="o",
    label="DEC active factors",
)
plt.plot(
    scaling_df["N"],
    scaling_df["N"],
    linestyle="--",
    label="full conditioning",
)
plt.xlabel("Total nonlinear-PDE factors N")
plt.ylabel("Factors evaluated")
plt.title("Decision complexity stays local")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.2,4.7))
plt.plot(
    scaling_df["N"],
    scaling_df["sparse_ESS"],
    marker="o",
    label="DEC sparse target",
)
plt.plot(
    scaling_df["N"],
    scaling_df["full_ESS"],
    marker="s",
    label="full nonlinear-PDE target",
)
plt.xlabel("Total nonlinear-PDE factors N")
plt.ylabel("GP-reference IS ESS fraction")
plt.title("DEC preserves an easy inference problem")
plt.legend()
plt.tight_layout()
plt.show()



## 8. Interpretation

This nonlinear example is a stronger gate than the linear-PDE experiment:

- the PDE residual itself contains \(\sin f\);
- overlapping factors remain present;
- the conditioned law is non-Gaussian;
- ordinary EI is used;
- the structural influence bound survives because the nonlinear residual derivatives are globally bounded;
- the sparse target remains easy for simple importance sampling;
- the full target rapidly leaves the GP-reference importance-sampling regime as the global PDE domain grows.

The main limitation is deliberate: this is still a globally weakly interacting / strongly stable nonlinear system. Allen–Cahn-like cubic residuals would require a bounded-state or high-probability truncation argument because their derivatives are not globally bounded.


In [ ]:

history_df.to_csv(
    OUT/"adaptive_history.csv",
    index=False,
)

scaling_df.to_csv(
    OUT/"scaling_results.csv",
    index=False,
)

print("Saved outputs to:",OUT)
